# Treino do NanoDet para seguir a trena

Modelo **NanoDet-Plus-m 320** (ShuffleNetV2 1.0x + GhostPAN), uma classe: `trena`. A entrada 320x320 é a versão pensada para a Raspberry.

O aumento acontece no treino, junto com a caixa. Validação e teste só normalizam a imagem.

No Colab, use um runtime com GPU. O dataset fica em `../get_data/dataset` quando o notebook roda de dentro de `train`. Ajuste `DATASET_DIR` se a pasta estiver em outro lugar. Com cerca de 200 imagens, as 100 épocas são o treino inteiro.


## 1. Ambiente

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# O notebook fica em train/. O COCO fica em get_data/dataset.
# No Colab, aponte DATASET_DIR para a pasta enviada.
ROOT = Path.cwd()
if (ROOT / "get_data" / "dataset").exists():
    DATASET_DIR = ROOT / "get_data" / "dataset"
else:
    DATASET_DIR = ROOT.parent / "get_data" / "dataset"
NANODET_DIR = Path(os.environ.get("NANODET_DIR", ROOT / "third_party" / "nanodet"))

if not (DATASET_DIR / "annotations" / "instances.json").exists():
    raise FileNotFoundError(
        f"Não achei {DATASET_DIR / 'annotations' / 'instances.json'}. "
        "Ajuste DATASET_DIR."
    )

if not NANODET_DIR.exists():
    NANODET_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/RangiLyu/nanodet.git", str(NANODET_DIR)]
    )

# O requirements.txt do NanoDet prende torch<2 e derruba o PyTorch
# que já está neste ambiente. Instala só o Lightning 1.9, que o
# código do treino ainda usa, e o pacote sem dependências.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "pytorch-lightning==1.9.5",
        "torchmetrics==0.11.4",
        "imagesize",
        "pyaml",
    ]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"],
    cwd=NANODET_DIR,
)

# torch._six foi removido no PyTorch 2. O collate do NanoDet ainda importa isso.
collate_path = NANODET_DIR / "nanodet" / "data" / "collate.py"
collate_src = collate_path.read_text(encoding="utf-8")
old_import = "from torch._six import string_classes"
if old_import in collate_src:
    collate_path.write_text(
        collate_src.replace(old_import, "string_classes = (str, bytes)"),
        encoding="utf-8",
    )
    print("collate.py ajustado para PyTorch 2")

import torch

def cuda_ready():
    try:
        return torch.cuda.is_available() and torch.cuda.device_count() > 0
    except Exception as exc:
        print("CUDA indisponível:", exc)
        return False

USE_GPU = cuda_ready()
print("dataset:", DATASET_DIR)
print("nanodet:", NANODET_DIR)
print("cuda:", torch.cuda.get_device_name(0) if USE_GPU else "cpu")
print("Se o import do nanodet falhar, reinicie o runtime e rode de novo a partir da célula do dataset.")


## 2. Dataset

In [ ]:
import json
import random

import cv2
import matplotlib.pyplot as plt

ANN_PATH = DATASET_DIR / "annotations" / "instances.json"
IMG_DIR = DATASET_DIR / "images"
coco = json.loads(ANN_PATH.read_text(encoding="utf-8"))

images = coco["images"]
anns_by_image = {}
for ann in coco["annotations"]:
    anns_by_image.setdefault(ann["image_id"], []).append(ann)

print(f"{len(images)} imagens, {len(coco['annotations'])} caixas, classes: {coco['categories']}")

missing = [img["file_name"] for img in images if not (IMG_DIR / img["file_name"]).exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} imagens do JSON não estão em {IMG_DIR}. Exemplo: {missing[:3]}"
    )

preview = images[:4]
fig, axes = plt.subplots(1, len(preview), figsize=(4 * len(preview), 4))
if len(preview) == 1:
    axes = [axes]
for ax, img_meta in zip(axes, preview):
    frame = cv2.cvtColor(cv2.imread(str(IMG_DIR / img_meta["file_name"])), cv2.COLOR_BGR2RGB)
    for ann in anns_by_image.get(img_meta["id"], []):
        x, y, w, h = ann["bbox"]
        cv2.rectangle(frame, (int(x), int(y)), (int(x + w), int(y + h)), (0, 220, 0), 2)
    ax.imshow(frame)
    ax.set_title(img_meta["file_name"])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Divisão treino, validação e teste

70% / 15% / 15%, semente fixa. As fotos continuam em `dataset/images`. Os três JSON só listam o subconjunto. O teste não entra no treino nem na escolha do melhor checkpoint.


In [ ]:
SEED = 42
random.seed(SEED)
shuffled = images.copy()
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
splits = {
    "train": shuffled[:n_train],
    "val": shuffled[n_train:n_train + n_val],
    "test": shuffled[n_train + n_val:],
}
if min(len(part) for part in splits.values()) < 1:
    raise RuntimeError(
        f"Dataset pequeno demais para três partes: {[len(p) for p in splits.values()]}"
    )

ann_dir = DATASET_DIR / "annotations"
for name, part in splits.items():
    ids = {img["id"] for img in part}
    payload = {
        "images": part,
        "annotations": [ann for ann in coco["annotations"] if ann["image_id"] in ids],
        "categories": coco["categories"],
    }
    path = ann_dir / f"{name}.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{name}: {len(part)} imagens -> {path.name}")


## 4. Aumento para a navegação

O robô vê a trena longe e perto, fora do centro, à esquerda ou à direita, com a câmera um pouco inclinada e com a luz mudando. Isso entra no pipeline de treino do NanoDet, que desloca a caixa junto com a imagem. Validação e teste não recebem esse aumento.

A célula seguinte grava o config e desenha amostras já passadas por esse pipeline, para conferir a caixa antes do treino.


In [ ]:
import yaml
import torch

CONFIG_DIR = ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "trena_nanodet_plus_m320.yml"
CKPT_PATH = ROOT / "weights" / "nanodet-plus-m_320_checkpoint.ckpt"
SAVE_DIR = ROOT / "workspace" / "trena_nanodet"

MEAN_STD = [[103.53, 116.28, 123.675], [57.375, 57.12, 58.395]]
TRAIN_PIPELINE = {
    "perspective": 0.0,
    "scale": [0.4, 1.6],
    "stretch": [[0.85, 1.15], [0.85, 1.15]],
    "rotation": 15,
    "shear": 0.0,
    "translate": 0.25,
    "flip": 0.5,
    "brightness": 0.4,
    "contrast": [0.5, 1.5],
    "saturation": [0.4, 1.4],
    "normalize": MEAN_STD,
}
VAL_PIPELINE = {"normalize": MEAN_STD}
gpu_ids = [0] if USE_GPU else -1

config = {
    "save_dir": str(SAVE_DIR),
    "model": {
        "weight_averager": {"name": "ExpMovingAverager", "decay": 0.9998},
        "arch": {
            "name": "NanoDetPlus",
            "detach_epoch": 10,
            "backbone": {
                "name": "ShuffleNetV2",
                "model_size": "1.0x",
                "out_stages": [2, 3, 4],
                "activation": "LeakyReLU",
            },
            "fpn": {
                "name": "GhostPAN",
                "in_channels": [116, 232, 464],
                "out_channels": 96,
                "kernel_size": 5,
                "num_extra_level": 1,
                "use_depthwise": True,
                "activation": "LeakyReLU",
            },
            "head": {
                "name": "NanoDetPlusHead",
                "num_classes": 1,
                "input_channel": 96,
                "feat_channels": 96,
                "stacked_convs": 2,
                "kernel_size": 5,
                "strides": [8, 16, 32, 64],
                "activation": "LeakyReLU",
                "reg_max": 7,
                "norm_cfg": {"type": "BN"},
                "loss": {
                    "loss_qfl": {
                        "name": "QualityFocalLoss",
                        "use_sigmoid": True,
                        "beta": 2.0,
                        "loss_weight": 1.0,
                    },
                    "loss_dfl": {"name": "DistributionFocalLoss", "loss_weight": 0.25},
                    "loss_bbox": {"name": "GIoULoss", "loss_weight": 2.0},
                },
            },
            "aux_head": {
                "name": "SimpleConvHead",
                "num_classes": 1,
                "input_channel": 192,
                "feat_channels": 192,
                "stacked_convs": 4,
                "strides": [8, 16, 32, 64],
                "activation": "LeakyReLU",
                "reg_max": 7,
            },
        },
    },
    "data": {
        "train": {
            "name": "CocoDataset",
            "img_path": str(IMG_DIR),
            "ann_path": str(ann_dir / "train.json"),
            "input_size": [320, 320],
            "keep_ratio": False,
            "pipeline": TRAIN_PIPELINE,
        },
        "val": {
            "name": "CocoDataset",
            "img_path": str(IMG_DIR),
            "ann_path": str(ann_dir / "val.json"),
            "input_size": [320, 320],
            "keep_ratio": False,
            "pipeline": VAL_PIPELINE,
        },
    },
    "device": {
        "gpu_ids": gpu_ids,
        "workers_per_gpu": 2,
        "batchsize_per_gpu": 16,
        "precision": 32,
    },
    "schedule": {
        "load_model": str(CKPT_PATH),
        "optimizer": {"name": "AdamW", "lr": 0.0002, "weight_decay": 0.05},
        "warmup": {"name": "linear", "steps": 50, "ratio": 0.001},
        "total_epochs": 100,
        "lr_schedule": {
            "name": "CosineAnnealingLR",
            "T_max": 100,
            "eta_min": 0.000001,
        },
        "val_intervals": 5,
    },
    "grad_clip": 35,
    "evaluator": {"name": "CocoDetectionEvaluator", "save_key": "AP_50"},
    "log": {"interval": 10},
    "class_names": ["trena"],
}

CONFIG_PATH.write_text(
    yaml.safe_dump(config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)
print("config:", CONFIG_PATH)


In [ ]:
import numpy as np
from nanodet.data.dataset import build_dataset
from nanodet.util import cfg, load_config

load_config(cfg, str(CONFIG_PATH))
train_ds = build_dataset(cfg.data.train, "train")
mean = np.array(MEAN_STD[0], dtype=np.float32)
std = np.array(MEAN_STD[1], dtype=np.float32)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
shown = 0
cursor = 0
while shown < 4 and cursor < len(train_ds) * 3:
    sample = train_ds[cursor % len(train_ds)]
    cursor += 1
    if sample is None or len(sample.get("gt_bboxes", [])) == 0:
        continue
    img = sample["img"]
    if isinstance(img, torch.Tensor):
        img = img.detach().cpu().numpy()
    # CHW normalizado em BGR -> HWC RGB para o desenho.
    if img.ndim == 3 and img.shape[0] == 3:
        vis = np.transpose(img, (1, 2, 0)) * std + mean
    else:
        vis = img * std + mean
    vis = np.clip(vis, 0, 255).astype(np.uint8)
    vis = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    for box in np.asarray(sample["gt_bboxes"]):
        x1, y1, x2, y2 = [int(v) for v in box[:4]]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 220, 0), 2)
    axes[shown].imshow(vis)
    axes[shown].set_title(f"aug {shown + 1}")
    axes[shown].axis("off")
    shown += 1
plt.tight_layout()
plt.show()


## 5. Backbone e treino

O checkpoint é o NanoDet-Plus-m 320 treinado no COCO. O carregador oficial fica com o ShuffleNetV2 e ignora a cabeça de 80 classes, que não tem o mesmo formato da cabeça de 1 classe.

O melhor arquivo é `workspace/trena_nanodet/model_best/model_best.ckpt`, escolhido pelo AP50 na validação.


In [ ]:
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not CKPT_PATH.exists():
    url = (
        "https://github.com/RangiLyu/nanodet/releases/download/"
        "v1.0.0-alpha-1/nanodet-plus-m_320_checkpoint.ckpt"
    )
    subprocess.check_call(["curl", "-L", "-o", str(CKPT_PATH), url])
print("backbone:", CKPT_PATH, "tamanho MB", round(CKPT_PATH.stat().st_size / 1e6, 1))


In [ ]:
collate_path = NANODET_DIR / "nanodet" / "data" / "collate.py"
collate_src = collate_path.read_text(encoding="utf-8")
old_import = "from torch._six import string_classes"
if old_import in collate_src:
    collate_path.write_text(
        collate_src.replace(old_import, "string_classes = (str, bytes)"),
        encoding="utf-8",
    )

subprocess.check_call(
    [sys.executable, "tools/train.py", str(CONFIG_PATH), "--seed", "42"],
    cwd=NANODET_DIR,
)
BEST_CKPT = SAVE_DIR / "model_best" / "model_best.ckpt"
if not BEST_CKPT.exists():
    raise FileNotFoundError(f"Treino terminou sem {BEST_CKPT}")
print("melhor checkpoint:", BEST_CKPT)
print((SAVE_DIR / "model_best" / "eval_results.txt").read_text(encoding="utf-8")[-2000:])


## 6. Métricas no teste

O `tools/test.py` avalia o conjunto apontado em `data.val`. Aqui esse caminho passa a ser o `test.json`, que ficou de fora do treino e da escolha do checkpoint. As figuras são previsões nesse conjunto.


In [ ]:
test_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
test_config["data"]["val"]["ann_path"] = str(ann_dir / "test.json")
test_config["save_dir"] = str(ROOT / "workspace" / "trena_nanodet_test")
TEST_CONFIG = CONFIG_DIR / "trena_nanodet_plus_m320_test.yml"
TEST_CONFIG.write_text(
    yaml.safe_dump(test_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

subprocess.check_call(
    [
        sys.executable,
        "tools/test.py",
        "--config",
        str(TEST_CONFIG),
        "--model",
        str(BEST_CKPT),
        "--task",
        "val",
    ],
    cwd=NANODET_DIR,
)

reports = sorted((ROOT / "workspace" / "trena_nanodet_test").glob("*/eval_results.txt"))
if not reports:
    raise FileNotFoundError("eval_results.txt não foi gerado")
print(reports[-1].read_text(encoding="utf-8"))


In [ ]:
sys.path.insert(0, str(NANODET_DIR))
from demo.demo import Predictor
from nanodet.util import Logger, cfg, load_config

load_config(cfg, str(TEST_CONFIG))
device = "cuda:0" if USE_GPU else "cpu"
predictor = Predictor(cfg, str(BEST_CKPT), Logger(0, use_tensorboard=False), device=device)

test_images = json.loads((ann_dir / "test.json").read_text(encoding="utf-8"))["images"]
sample_images = test_images[:4]
out_dir = ROOT / "workspace" / "trena_nanodet_test" / "preview"
out_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(sample_images), figsize=(4 * len(sample_images), 4))
if len(sample_images) == 1:
    axes = [axes]
for ax, img_meta in zip(axes, sample_images):
    path = IMG_DIR / img_meta["file_name"]
    meta, res = predictor.inference(str(path))
    drawn = predictor.model.head.show_result(
        meta["raw_img"][0],
        res[0],
        cfg.class_names,
        score_thres=0.35,
        show=False,
    )
    cv2.imwrite(str(out_dir / img_meta["file_name"]), drawn)
    ax.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    ax.set_title(img_meta["file_name"])
    ax.axis("off")
plt.tight_layout()
plt.show()
print("prévias em", out_dir)
